# XSIGHT RetailAction X-CLIP smoke test
Run in Kaggle with Internet ON. Requires 25 clips and manifest.json already present in `/kaggle/working/retailaction_smoke/` (not included). Real pretrained inference, not trained shoplifting detection.

In [ ]:
import json, pathlib, subprocess, sys
root=pathlib.Path("/kaggle/working/retailaction_smoke")
print("Videos:",len(list(root.glob("*.mp4"))),"Manifest:",(root/"manifest.json").exists())
assert (root/"manifest.json").exists(), "Missing clips/manifest: re-run original extraction notebook first or attach them as a Kaggle dataset."
subprocess.check_call([sys.executable,"-m","pip","-q","install","transformers>=4.46,<5","huggingface-hub>=0.30,<2","opencv-python-headless","safetensors"])
import cv2, numpy as np, torch
from transformers import XCLIPModel,XCLIPProcessor
from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,confusion_matrix,classification_report
raw=json.loads((root/"manifest.json").read_text())
rows=raw if isinstance(raw,list) else next((raw[k] for k in ("samples","actions","clips","items","records") if isinstance(raw.get(k),list)),None)
assert rows, "No labeled rows in manifest"
labels=["take","put","touch"]
prompts=["A person takes an item from a store shelf.","A person puts an item onto a store shelf.","A person touches an item on a store shelf."]
def video_path(row):
    v=next((row[k] for k in ("video","video_path","path","file") if row.get(k)),None)
    assert v, "Missing video path"
    p=pathlib.Path(v)
    return p if p.exists() else root/p.name
def frames_for(path,n=8):
    cap=cv2.VideoCapture(str(path)); count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    assert cap.isOpened(),f"Cannot open {path}"
    frames=[]
    for i in np.linspace(0,max(count-1,0),n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(i)); ok,bgr=cap.read()
        if ok: frames.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    cap.release()
    assert frames,f"No decoded frames: {path}"
    while len(frames)<n:frames.append(frames[-1].copy())
    return frames[:n]
device="cuda" if torch.cuda.is_available() else "cpu"
model_id="microsoft/xclip-base-patch32"
processor=XCLIPProcessor.from_pretrained(model_id)
model=XCLIPModel.from_pretrained(model_id).to(device).eval()
print("Model:",model_id,"Device:",device,"Rows:",len(rows))
predictions=[]; errors=[]
for i,row in enumerate(rows):
    try:
        frames=frames_for(video_path(row))
        pixels=processor.image_processor(images=frames,return_tensors="pt")["pixel_values"]
        if pixels.ndim==4:pixels=pixels.unsqueeze(0)
        assert pixels.ndim==5, f"Unexpected pixel shape: {tuple(pixels.shape)}"
        text=processor.tokenizer(prompts,padding=True,truncation=True,return_tensors="pt")
        inputs={**text,"pixel_values":pixels}
        with torch.inference_mode():
            out=model(**{k:v.to(device) for k,v in inputs.items()})
            scores=out.logits_per_video.softmax(dim=-1)[0].cpu().numpy()
        predictions.append({"sample":str(row.get("sample",i)),"label":str(row["label"]).lower(),"prediction":labels[int(scores.argmax())],"scores":dict(zip(labels,map(float,scores)))})
    except Exception as exc:errors.append({"index":i,"error":repr(exc)})
    (root/"xclip_predictions.json").write_text(json.dumps({"predictions":predictions,"errors":errors},indent=2))
    print(f"{i+1}/{len(rows)}",predictions[-1]["prediction"] if len(predictions)==i+1 else "ERROR")
valid=[r for r in predictions if r["label"] in labels]
assert valid,"No valid predictions; inspect errors"
yt=[r["label"] for r in valid];yp=[r["prediction"] for r in valid]
report={"dataset":"standard-cognition/RetailAction","experiment":"pretrained_xclip_smoke","model":model_id,"evaluated":len(valid),"errors":errors,"accuracy":accuracy_score(yt,yp),"balanced_accuracy":balanced_accuracy_score(yt,yp),"macro_f1":f1_score(yt,yp,labels=labels,average="macro",zero_division=0),"class_order":labels,"confusion_matrix":confusion_matrix(yt,yp,labels=labels).tolist(),"caveat":"Selected smoke clips; not validated production performance or shoplifting detection"}
(root/"xclip_report.json").write_text(json.dumps(report,indent=2))
print(classification_report(yt,yp,labels=labels,zero_division=0))
print(json.dumps(report,indent=2))
